In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
from pathlib import Path
from scipy.spatial import cKDTree
import seaborn as sns
from utils import construct_path, read_feather_gcs, pq_to_xy

pd.set_option('display.max_rows', 300)

/home/guardomayas/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Packages loaded successfully


In [2]:
data_dir = Path("connectome_data/processed")
col_nodes = pd.read_feather(data_dir / "columnar_nodes.feather")
T4_T5_nodes = pd.read_feather(data_dir / "T4_T5_home_columns.feather")
ct1_edgelist = pd.read_feather(data_dir / "ct1_edgelist_full.feather")
print(ct1_edgelist["count"].min())

1


In [3]:
nodes_df = pd.concat([col_nodes, T4_T5_nodes], ignore_index=True).drop_duplicates("malecns_09_id", keep="last").copy()
nodes_df

,malecns_09_id,side,cell_type,neurotransmitter_predicted,neurotransmitter_score,hex_1,hex_2,p,q
0,100043,right,C2,gaba,0.528316,24.0,17.0,6.0,-2.0
1,100062,right,C2,unclear,0.398571,24.0,35.0,6.0,16.0
2,100134,right,C2,unclear,0.405942,22.0,31.0,4.0,12.0
3,100156,right,C2,unclear,0.467389,28.0,38.0,10.0,19.0
4,100229,right,C2,unclear,0.581083,27.0,13.0,9.0,-6.0
...,...,...,...,...,...,...,...,...,...
30298,139284,right,T5d,unclear,0.974427,7.0,2.0,-11.0,-17.0
30299,163871,right,T5d,unclear,0.974427,31.0,39.0,13.0,20.0
30300,142354,right,T5d,unclear,0.974427,34.0,21.0,16.0,2.0
30301,160570,right,T5d,unclear,0.974427,27.0,39.0,9.0,20.0


## Add offset to ct1_edges and filter out noncolumnar

In [4]:
post_home = nodes_df[
    [
        "malecns_09_id",
        "hex_1",
        "hex_2",
        "p",
        "q",
    ]
].rename(columns={
    "malecns_09_id": "post",
    "hex_1": "post_hex_1",
    "hex_2": "post_hex_2",
    "p": "post_p",
    "q": "post_q",
})

ct1_edges_annot = ct1_edgelist.merge(
    post_home,
    on="post",
    how="left"
)

pre_home = nodes_df[
    [
        "malecns_09_id",
        "hex_1",
        "hex_2",
        "p",
        "q",
    ]
].rename(columns={
    "malecns_09_id": "pre",
    "hex_1": "pre_hex_1",
    "hex_2": "pre_hex_2",
    "p": "pre_p",
    "q": "pre_q",
})

ct1_edges_annot = ct1_edges_annot.merge(
    pre_home,
    on="pre",
    how="left"
)

In [5]:
ct1_edges_annot = ct1_edges_annot.query("count > 3").copy().reset_index(drop=True)
ct1_edges_annot["dp_syn_to_post"] = (
    ct1_edges_annot["syn_p"] - ct1_edges_annot["post_p"]
)

ct1_edges_annot["dq_syn_to_post"] = (
    ct1_edges_annot["syn_q"] - ct1_edges_annot["post_q"]
)

ct1_edges_annot["dp_syn_to_pre"] = (
    ct1_edges_annot["syn_p"] - ct1_edges_annot["pre_p"]
)

ct1_edges_annot["dq_syn_to_pre"] = (
    ct1_edges_annot["syn_q"] - ct1_edges_annot["pre_q"]
)

ct1_edges_annot["ct1_direction"] = np.select(
    [
        ct1_edges_annot["pre_cell_type"].eq("CT1"),
        ct1_edges_annot["post_cell_type"].eq("CT1"),
    ],
    [
        "CT1_output",
        "CT1_input",
    ],
    default="not_CT1",
)


In [ ]:
ct1_final_edges = ct1_edges_annot.copy()

is_output = ct1_final_edges["ct1_direction"].eq("CT1_output")
is_input  = ct1_final_edges["ct1_direction"].eq("CT1_input")

# Source location
ct1_final_edges["source_hex_1"] = np.where(
    is_output,
    ct1_final_edges["syn_hex_1"],
    ct1_final_edges["pre_hex_1"],
)

ct1_final_edges["source_hex_2"] = np.where(
    is_output,
    ct1_final_edges["syn_hex_2"],
    ct1_final_edges["pre_hex_2"],
)

ct1_final_edges["source_p"] = np.where(
    is_output,
    ct1_final_edges["syn_p"],
    ct1_final_edges["pre_p"],
)

ct1_final_edges["source_q"] = np.where(
    is_output,
    ct1_final_edges["syn_q"],
    ct1_final_edges["pre_q"],
)

# Target/reference location
ct1_final_edges["target_hex_1"] = np.where(
    is_output,
    ct1_final_edges["post_hex_1"],
    ct1_final_edges["syn_hex_1"],
)

ct1_final_edges["target_hex_2"] = np.where(
    is_output,
    ct1_final_edges["post_hex_2"],
    ct1_final_edges["syn_hex_2"],
)

ct1_final_edges["target_p"] = np.where(
    is_output,
    ct1_final_edges["post_p"],
    ct1_final_edges["syn_p"],
)

ct1_final_edges["target_q"] = np.where(
    is_output,
    ct1_final_edges["post_q"],
    ct1_final_edges["syn_q"],
)

ct1_final_edges["dp"] = (
    ct1_final_edges["source_p"] - ct1_final_edges["target_p"]
)

ct1_final_edges["dq"] = (
    ct1_final_edges["source_q"] - ct1_final_edges["target_q"]
)

final_cols = [
    "pre",
    "post",
    "pre_cell_type",
    "post_cell_type",
    "ct1_direction",
    "count",
    "neuropil",

    "source_hex_1",
    "source_hex_2",
    "source_p",
    "source_q",

    "target_hex_1",
    "target_hex_2",
    "target_p",
    "target_q",

    "dp",
    "dq",

    # # "syn_column_id",
    # "syn_hex_1",
    # "syn_hex_2",
    # "syn_p",
    # "syn_q",
    "syn_top_nt",
    "mean_syn_top_p",
]

ct1_final_edges = ct1_final_edges[final_cols].rename(columns={
    "syn_top_nt": "edge_nt",
    "mean_syn_top_p": "edge_nt_score",
})

ct1_final_edges["count"].min() #Not an effect of thresholding, 
                            # but just including only columnar and T4/T5 nodes

np.int64(4)

In [7]:
ct1_final_edges.to_feather(data_dir / "ct1_columnar_edges.feather", compression="uncompressed")